In [1]:
import os
os.chdir('../..')

In [2]:
import selfies as sf
from scipy.spatial.distance import cosine, euclidean
from src.datasets import QM9Dataset
from src.descriptors import SOAP, ACSF
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import polars as pl
from rdkit import DataStructs, Chem
from rdkit.Chem import AllChem
from src.clusters import ClusterAnalysis

/Users/karlfindhansen/Desktop/DTU/Kandidat/Thesis/Anomaly-Detection-in-Molecular-and-Materials-Datasets/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
qm9 = QM9Dataset()
qm9.load()
qm9.add_morgan_fingerprints()
qm9.add_selfies_onehot()
qm9.add_selfies_transformer()
qm9.add_soap()
qm9.add_acsf()

2026-02-15 09:16:09.303 | INFO     | src.datasets:load:74 - Loading QM9 from data/QM9/dataset_cleaned.csv...
2026-02-15 09:16:09.312 | INFO     | src.features:compute_morgan_fingerprints:65 - Computing Morgan Fingerprints (Radius=3, Size=2048)...
2026-02-15 09:16:11.929 | INFO     | src.features:compute_selfies_onehot:102 - Computing One-Hot Encodings...
2026-02-15 09:16:12.053 | INFO     | src.features:compute_selfies_transformer:77 - Computing Transformer Embeddings (seyonec/ChemBERTa-zinc-base-v1)...
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 772.67it/s, Materializing param=pooler.dense.weight]                             
2026-02-15 09:16:25.603 | INFO     | src.features:compute_soap:125 - Computing SOAP (rcut=6.0, nmax=8, lmax=6)...
2026-02-15 09:16:33.951 | SUCCESS  | src.datasets:add_soap:193 - Added SOAP embeddings.
2026-02-15 09:16:33.952 | INFO     | src.features:compute_acsf:155 - Computing ACSF (rcut=6.0)...
2026-02-15 09:16:40.577 | SUCCESS  | src.datasets:add

In [4]:
def add_cluster_label_to_df(df):
    
    true_labels = qm9.df['structure_class']
    num_clusters = len(set(true_labels))

    methods = ["morgan_fingerprint", "selfies_transformer", "selfies_onehot"]
    new_columns = []

    for method in methods:
        
        X = np.array(qm9.df[method].to_list(), dtype=np.float32)

        if len(X.shape) > 2: 
            X = X.reshape(X.shape[0], -1)

        analyzer = ClusterAnalysis(X, true_labels=true_labels, meta_df=qm9.df) 

        kmeans_labels = analyzer.run(method='kmeans', n_clusters=num_clusters)
        new_columns.append(pl.Series(f"{method}_kmeans", kmeans_labels))
        overlap_score, is_overlap = analyzer.calculate_overlap()
        new_columns.append(pl.Series(f"{method}_overlap_score", overlap_score))
        new_columns.append(pl.Series(f"{method}_is_overlapping", is_overlap))

        dbscan_labels = analyzer.run(method='dbscan', n_clusters=num_clusters)
        new_columns.append(pl.Series(f"{method}_dbscan", dbscan_labels))
        overlap_score, is_overlap = analyzer.calculate_overlap()
        new_columns.append(pl.Series(f"{method}_overlap_score", overlap_score))
        new_columns.append(pl.Series(f"{method}_is_overlapping", is_overlap))

        hierarchical_labels = analyzer.run(method='hierarchical', n_clusters=num_clusters)
        new_columns.append(pl.Series(f"{method}_hierarchical", hierarchical_labels))
        overlap_score, is_overlap = analyzer.calculate_overlap()
        new_columns.append(pl.Series(f"{method}_overlap_score", overlap_score))
        new_columns.append(pl.Series(f"{method}_is_overlapping", is_overlap))

    df = df.with_columns(new_columns)

    return df

In [5]:
qm9.df = add_cluster_label_to_df(qm9.df)

--- Running KMEANS ---


TypeError: Series.mean() got an unexpected keyword argument 'axis'